[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Track a Moving Camera with BoT-SORT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/trackers/blob/develop/docs/cookbooks/how-to-track-a-moving-camera-with-botsort.ipynb)

A tracker predicts where each person will be in the next frame and matches the new detections to those predictions. When the camera moves, everything in the image shifts with it, so a prediction made from the last frame lands beside the person instead of on them. The match fails and the person gets a new ID.

BoT-SORT handles this with camera motion compensation (CMC): before matching, it estimates how the camera moved between the two frames and moves every prediction by the same amount.

In this notebook you will track MOT17-13, filmed from a moving bus, with CMC off and on, watch one person through the moment the two runs disagree, and measure the difference on all seven MOT17 validation clips.

## Setup

### Install dependencies

`gdown` downloads the YOLOX detections used by the MOT17 benchmark. Everything in this notebook runs on a CPU runtime.

In [ ]:
!pip install -q trackers gdown

### Download MOT17 and the detections

The validation split holds the second half of each MOT17 training sequence, with ground truth. The YOLOX detections are the ones the [benchmark results](https://trackers.roboflow.com/latest/evaluations/results/) use, so both runs see exactly the same boxes and any difference comes from the tracker.

In [ ]:
from pathlib import Path

import numpy as np

ROOT = Path("mot17-cmc")
MOT17_VAL = ROOT / "mot17" / "val"
DETECTIONS_DIR = ROOT / "detections"
DETECTIONS_DIR.mkdir(parents=True, exist_ok=True)
MOVING = ["MOT17-05-FRCNN", "MOT17-10-FRCNN", "MOT17-11-FRCNN", "MOT17-13-FRCNN"]
STILL = ["MOT17-02-FRCNN", "MOT17-04-FRCNN", "MOT17-09-FRCNN"]

!trackers download --name mot17 --split val --asset annotations,frames --output {ROOT}
!gdown 1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT -O {ROOT}/yolox.zip
!unzip -qo {ROOT}/yolox.zip -d {ROOT}/yolox

for sequence in MOVING + STILL:
    rows = np.loadtxt(ROOT / "yolox" / "val" / f"{sequence.replace('-FRCNN', '')}_val.txt", delimiter=",", ndmin=2)
    rows = rows[rows[:, 5] > 0]
    frames = rows[:, 0] - rows[:, 0].min() + 1
    x1, y1, x2, y2, score = rows[:, 1:6].T
    mot = np.column_stack([frames, -np.ones_like(frames), x1, y1, x2 - x1, y2 - y1, score])
    np.savetxt(DETECTIONS_DIR / f"{sequence}.txt", mot, delimiter=",", fmt=["%d", "%d", "%.2f", "%.2f", "%.2f", "%.2f", "%.4f"])

## Track MOT17-13 with CMC off and on

The two runs differ in one setting, `--tracker.enable_cmc`. CMC reads the frames to measure the camera motion, so `--source` points at the images even though the detections come from a file.

In [ ]:
OUTPUT = ROOT / "predictions"
SEQUENCE = "MOT17-13-FRCNN"

for run, enable_cmc in (("cmc_off", "false"), ("cmc_on", "true")):
    !trackers track --source {MOT17_VAL}/{SEQUENCE}/img1 --detection.mot_file {DETECTIONS_DIR}/{SEQUENCE}.txt \
        --tracker botsort --tracker.enable_cmc {enable_cmc} \
        --output.mot_results {OUTPUT}/{run}/{SEQUENCE}.txt --output.overwrite

## Watch one person

Person 28 in the ground truth walks along the pavement in plain view, around images 129 to 141, while the bus moves and the whole scene slides past. The crops below follow him and label the ID each run gives him: first with CMC off, then with CMC on.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from trackers.io.mot import load_mot_file

PERSON = 28
FRAMES = range(129, 142, 2)

ground_truth = np.loadtxt(MOT17_VAL / SEQUENCE / "gt" / "gt.txt", delimiter=",", ndmin=2)
person_boxes = {int(r[0]): r[2:6] for r in ground_truth if int(r[1]) == PERSON}


def show_strip(run):
    predictions = load_mot_file(OUTPUT / run / f"{SEQUENCE}.txt")
    figure, axes = plt.subplots(1, len(FRAMES), figsize=(3 * len(FRAMES), 4))
    for ax, frame_index in zip(axes, FRAMES):
        image = cv2.cvtColor(cv2.imread(str(MOT17_VAL / SEQUENCE / "img1" / f"{frame_index:06d}.jpg")), cv2.COLOR_BGR2RGB)
        x, y, w, h = person_boxes[frame_index]
        x0, y0 = int(max(x - w, 0)), int(max(y - 0.3 * h, 0))
        x1, y1 = int(min(x + 2 * w, image.shape[1])), int(min(y + 1.3 * h, image.shape[0]))
        ax.imshow(image[y0:y1, x0:x1])
        frame_data = predictions.get(frame_index)
        if frame_data is not None:
            for track_id, (bx, by, bw, bh) in zip(frame_data.ids, frame_data.boxes):
                overlap_x = min(bx + bw, x + w) - max(bx, x)
                overlap_y = min(by + bh, y + h) - max(by, y)
                if overlap_x > 0.5 * w and overlap_y > 0.5 * h:
                    ax.add_patch(plt.Rectangle((bx - x0, by - y0), bw, bh, fill=False, color="yellow", linewidth=2))
                    ax.set_title(f"image {frame_index}: #{int(track_id)}")
                    break
            else:
                ax.set_title(f"image {frame_index}: no track")
        ax.axis("off")
    figure.suptitle(run.replace("_", " ").upper())
    plt.tight_layout()
    plt.show()


show_strip("cmc_off")

In [ ]:
show_strip("cmc_on")

## How the camera step works

On every frame, BoT-SORT picks well-textured points in the previous frame, finds where they moved in the current one with sparse optical flow, and fits one transform to those movements. Points on people move on their own, so the fit rejects them as outliers and follows the background. The same transform then moves every track's predicted box before matching, so the predictions land where the people now are.

This only works when BoT-SORT gets the frame: pass it as `tracker.update(detections, frame=frame)`. Without a frame, the step is skipped for that update. If you already estimate the camera motion yourself, pass it as `cmc_transform` instead. See [BoT-SORT](https://trackers.roboflow.com/latest/trackers/botsort/) and the [Motion API](https://trackers.roboflow.com/latest/api/motion/) for the parameters.

## Measure it on all seven clips

MOT17's validation clips include four filmed from a moving camera and three from a fixed one. Track the rest of them with CMC off and on, then score the moving and the still clips separately. On the moving clips, ID switches should drop sharply with CMC on; on the still clips there is little camera motion to correct, so the two runs should stay close.

In [ ]:
for sequence in MOVING + STILL:
    if sequence == SEQUENCE:
        continue
    for run, enable_cmc in (("cmc_off", "false"), ("cmc_on", "true")):
        !trackers track --source {MOT17_VAL}/{sequence}/img1 --detection.mot_file {DETECTIONS_DIR}/{sequence}.txt \
            --tracker botsort --tracker.enable_cmc {enable_cmc} \
            --output.mot_results {OUTPUT}/{run}/{sequence}.txt --output.overwrite

In [ ]:
for group, sequences in (("moving", MOVING), ("still", STILL)):
    seqmap = ROOT / f"seqmap_{group}.txt"
    seqmap.write_text("name\n" + "\n".join(sequences) + "\n")
    for run in ("cmc_off", "cmc_on"):
        print(f"{group} camera, {run}")
        !trackers eval --gt_dir {MOT17_VAL} --predictions_dir {OUTPUT}/{run} --seqmap {seqmap} --metrics '[CLEAR,HOTA,Identity]' --columns '[HOTA,IDF1,IDSW]'

You just tracked a moving camera with BoT-SORT and saw what camera motion compensation does. Nice work!

CMC is on by default in `BoTSORTTracker`, so in your own code the one thing to remember is to pass the frame to `update()`. To go further, the [BoT-SORT page](https://trackers.roboflow.com/latest/trackers/botsort/) covers its parameters, and [Track Objects](https://trackers.roboflow.com/latest/guides/track/) shows how to run it on video, a webcam or an RTSP stream.

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/trackers/issues).